In [37]:
import os
import shutil
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.metrics import roc_curve, auc, precision_recall_curve
import matplotlib.pyplot as plt
from tqdm import tqdm


In [38]:

# ----------------------------
# USER CONFIG
# ----------------------------
model_path = r"D:\FYP\Datasets\solar_cell_EL_image\solar_cell_EL_image\PVELAD\EL2021\othertypes\resnet50_good_defect.h5"
test_dir   = r"D:\FYP\Trial codes\resnet\pvelad_3class_aug_data_512\val\full"   # Can be a single folder with test images
save_dir   = r"D:\FYP\resnet_re_2"
img_size = (256, 256)


In [39]:

# ----------------------------
# LOAD MODEL
# ----------------------------
model = load_model(model_path)
print("\n✅ Model loaded successfully!\n")



✅ Model loaded successfully!



In [40]:

# ----------------------------
# LOAD IMAGES (handles unlabeled folder)
# ----------------------------
image_files = [f for f in os.listdir(test_dir)
                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'))]

if len(image_files) == 0:
    raise ValueError(f"No images found in: {test_dir}")

print(f"Found {len(image_files)} images in test folder.")


Found 404 images in test folder.


In [41]:

# ----------------------------
# MAKE SAVE FOLDERS (predicted class dirs)
# ----------------------------
class_names = ["good", "defect"]  # 🔹 Change these to your actual two class names
for cls in class_names:
    os.makedirs(os.path.join(save_dir, cls), exist_ok=True)


In [42]:

# ----------------------------
# RUN PREDICTIONS
# ----------------------------
preds = []
for img_name in tqdm(image_files, desc="Predicting"):
    img_path = os.path.join(test_dir, img_name)
    img = image.load_img(img_path, target_size=img_size)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0
    
    prob = model.predict(img_array, verbose=0)[0][0]
    pred_class = 1 if prob > 0.1 else 0  # Binary threshold
    preds.append(prob)
    
    # Copy image to predicted folder
    dest_path = os.path.join(save_dir, class_names[pred_class], img_name)
    shutil.copy(img_path, dest_path)

print(f"\n✅ All predictions saved under:\n   {save_dir}")



Predicting: 100%|██████████| 404/404 [00:49<00:00,  8.14it/s]


✅ All predictions saved under:
   D:\FYP\resnet_re_2


In [43]:
from ultralytics import YOLO
import os

# --- Paths ---
model_path = r"D:\FYP\Trial codes\runs\detect\100epochs_512res_yolov8s_preprocessing_class_balancing\weights\best.pt"  # your saved YOLOv8 model
dataset_path = r"D:\FYP\EL_Cleaned\bad\defect" # folder containing images
output_dir = r"D:\FYP\EL_Cleaned\bad\final_output"   # folder to save predictions
 # folder to save predictions

# --- Load model ---
model = YOLO(model_path)

# --- Run inference ---
results = model.predict(
    source=dataset_path,   # can be a folder, image, video, or glob (*.jpg)
    save=True,             # save annotated predictions
    save_txt=True,         # save YOLO-format txt labels (optional)
    conf=0.25,             # confidence threshold (adjust as needed)
    iou=0.45,              # NMS IoU threshold
    project=output_dir,    # root output folder
    name="yolov8_results", # subfolder name
    exist_ok=True          # overwrite if already exists
)

print("✅ Inference completed!")
print(f"Results saved in: {os.path.join(output_dir, 'yolov8_results')}")



WARNING 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



RuntimeError: Given groups=1, weight of size [32, 3, 3, 3], expected input[1, 1, 512, 512] to have 3 channels, but got 1 channels instead